In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Data cleaning started")

Data cleaning started


In [4]:
customers = pd.read_csv(RAW_DIR / "customers.csv")
payments = pd.read_csv(RAW_DIR / "payments.csv")
subscriptions = pd.read_csv(RAW_DIR / "subscriptions.csv")
subscription_plans = pd.read_csv(RAW_DIR / "subscription_plans.csv")
viewing_activity = pd.read_csv(RAW_DIR / "viewing_activity.csv")
content = pd.read_csv(RAW_DIR / "content.csv")
support_tickets = pd.read_csv(RAW_DIR / "support_tickets.csv")
customer_feedback = pd.read_csv(RAW_DIR / "customer_feedback.csv")
churn_labels = pd.read_csv(RAW_DIR / "churn_labels.csv")

print("All datasets loaded successfully.")

All datasets loaded successfully.


In [5]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8048 entries, 0 to 8047
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   customer_id          8048 non-null   object 
 1   first_name           8048 non-null   object 
 2   last_name            8048 non-null   object 
 3   age                  7726 non-null   float64
 4   gender               8048 non-null   object 
 5   country              7639 non-null   object 
 6   state_or_region      0 non-null      float64
 7   city                 7564 non-null   object 
 8   registration_date    8048 non-null   object 
 9   acquisition_channel  8048 non-null   object 
 10  customer_segment     8048 non-null   object 
 11  preferred_language   7794 non-null   object 
dtypes: float64(2), object(10)
memory usage: 754.6+ KB


In [6]:
customers.isnull().sum()

customer_id               0
first_name                0
last_name                 0
age                     322
gender                    0
country                 409
state_or_region        8048
city                    484
registration_date         0
acquisition_channel       0
customer_segment          0
preferred_language      254
dtype: int64

In [7]:
# Drop column with 100% missing values
customers = customers.drop(columns=["state_or_region"])

# Fill numerical missing values with median
customers["age"] = customers["age"].fillna(customers["age"].median())

# Fill categorical missing values with mode
customers["country"] = customers["country"].fillna(customers["country"].mode()[0])
customers["city"] = customers["city"].fillna(customers["city"].mode()[0])
customers["preferred_language"] = customers["preferred_language"].fillna(
    customers["preferred_language"].mode()[0]
)

print("Missing values after cleaning:")
print(customers.isnull().sum())

Missing values after cleaning:
customer_id            0
first_name             0
last_name              0
age                    0
gender                 0
country                0
city                   0
registration_date      0
acquisition_channel    0
customer_segment       0
preferred_language     0
dtype: int64


In [8]:
# Verify customer data after cleaning

print("Shape:", customers.shape)

print("\nMissing values:")
print(customers.isnull().sum())

print("\nDuplicate rows:", customers.duplicated().sum())

Shape: (8048, 11)

Missing values:
customer_id            0
first_name             0
last_name              0
age                    0
gender                 0
country                0
city                   0
registration_date      0
acquisition_channel    0
customer_segment       0
preferred_language     0
dtype: int64

Duplicate rows: 48


In [9]:
# Check duplicate customer IDs
duplicate_ids = customers[customers.duplicated("customer_id", keep=False)]

print("Duplicate customer IDs:", duplicate_ids["customer_id"].nunique())
print("Rows involved:", len(duplicate_ids))

Duplicate customer IDs: 48
Rows involved: 96


In [10]:
# Remove duplicate customer records
customers = customers.drop_duplicates(
    subset=["customer_id"],
    keep="first"
).copy()

print("Shape after removing duplicates:", customers.shape)
print("Duplicate customer IDs after cleaning:",
      customers["customer_id"].duplicated().sum())

Shape after removing duplicates: (8000, 11)
Duplicate customer IDs after cleaning: 0


In [11]:
subscriptions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8875 entries, 0 to 8874
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   subscription_id          8875 non-null   object 
 1   customer_id              8875 non-null   object 
 2   plan_id                  8875 non-null   object 
 3   subscription_start_date  8875 non-null   object 
 4   subscription_end_date    875 non-null    object 
 5   subscription_status      8875 non-null   object 
 6   auto_renew               8875 non-null   bool   
 7   cancellation_date        1991 non-null   object 
 8   cancellation_reason      1991 non-null   object 
 9   monthly_price            8875 non-null   float64
dtypes: bool(1), float64(1), object(8)
memory usage: 632.8+ KB


In [12]:
print("Missing values in subscriptions:")
print(subscriptions.isnull().sum())

Missing values in subscriptions:
subscription_id               0
customer_id                   0
plan_id                       0
subscription_start_date       0
subscription_end_date      8000
subscription_status           0
auto_renew                    0
cancellation_date          6884
cancellation_reason        6884
monthly_price                 0
dtype: int64


In [13]:
print("Subscription status:")
print(subscriptions["subscription_status"].value_counts(dropna=False))

print("\nCancellation reason by status:")
print(
    subscriptions.groupby("subscription_status")["cancellation_reason"]
    .apply(lambda x: x.notna().sum())
)

Subscription status:
subscription_status
Active          6009
Cancelled       1991
Plan Changed     875
Name: count, dtype: int64

Cancellation reason by status:
subscription_status
Active             0
Cancelled       1991
Plan Changed       0
Name: cancellation_reason, dtype: int64


In [14]:
print("Date column data types:")
print(subscriptions[
    ["subscription_start_date",
     "subscription_end_date",
     "cancellation_date"]
].dtypes)

Date column data types:
subscription_start_date    object
subscription_end_date      object
cancellation_date          object
dtype: object


In [15]:
date_columns = [
    "subscription_start_date",
    "subscription_end_date",
    "cancellation_date"
]

for column in date_columns:
    subscriptions[column] = pd.to_datetime(
        subscriptions[column],
        errors="coerce"
    )

print(subscriptions[date_columns].dtypes)

subscription_start_date    datetime64[ns]
subscription_end_date      datetime64[ns]
cancellation_date          datetime64[ns]
dtype: object


In [16]:
print("Missing/invalid dates after conversion:")
print(subscriptions[
    ["subscription_start_date",
     "subscription_end_date",
     "cancellation_date"]
].isna().sum())

Missing/invalid dates after conversion:
subscription_start_date       0
subscription_end_date      8000
cancellation_date          6884
dtype: int64


In [17]:
print("Invalid date relationships:")

invalid_end = (
    subscriptions["subscription_end_date"].notna()
    & (subscriptions["subscription_end_date"]
       < subscriptions["subscription_start_date"])
)

invalid_cancel = (
    subscriptions["cancellation_date"].notna()
    & (subscriptions["cancellation_date"]
       < subscriptions["subscription_start_date"])
)

print("End date before start date:", invalid_end.sum())
print("Cancellation date before start date:", invalid_cancel.sum())

Invalid date relationships:
End date before start date: 0
Cancellation date before start date: 42


In [18]:
invalid_cancel_rows = subscriptions[
    subscriptions["cancellation_date"].notna()
    & (
        subscriptions["cancellation_date"]
        < subscriptions["subscription_start_date"]
    )
]

invalid_cancel_rows[
    [
        "subscription_id",
        "customer_id",
        "subscription_start_date",
        "cancellation_date",
        "subscription_status",
        "cancellation_reason"
    ]
]

,subscription_id,customer_id,subscription_start_date,cancellation_date,subscription_status,cancellation_reason
125,SUB0000126,CUST000113,2026-04-18,2025-12-09,Cancelled,Poor customer service experience
157,SUB0000158,CUST000143,2026-03-02,2025-12-27,Cancelled,Not enough content
301,SUB0000302,CUST000268,2026-03-02,2026-01-29,Cancelled,Found alternative platform
331,SUB0000332,CUST000296,2026-01-26,2025-12-07,Cancelled,Found alternative platform
539,SUB0000540,CUST000491,2026-06-28,2025-12-22,Cancelled,Poor customer service experience
735,SUB0000736,CUST000670,2026-01-03,2025-12-02,Cancelled,Too expensive
940,SUB0000941,CUST000859,2026-01-26,2025-10-08,Cancelled,Content quality decline
1024,SUB0001025,CUST000935,2026-02-24,2025-11-23,Cancelled,Content quality decline
1282,SUB0001283,CUST001173,2026-04-09,2025-11-17,Cancelled,Poor customer service experience
1443,SUB0001444,CUST001319,2026-03-16,2025-12-04,Cancelled,Found alternative platform


In [19]:
invalid_cancel = (
    subscriptions["cancellation_date"].notna()
    & (
        subscriptions["cancellation_date"]
        < subscriptions["subscription_start_date"]
    )
)

subscriptions.loc[invalid_cancel, "cancellation_date"] = pd.NaT

print(
    "Invalid cancellation dates fixed:",
    invalid_cancel.sum()
)

Invalid cancellation dates fixed: 42


In [20]:
invalid_cancel_check = (
    subscriptions["cancellation_date"].notna()
    & (
        subscriptions["cancellation_date"]
        < subscriptions["subscription_start_date"]
    )
)

print(
    "Invalid cancellation dates remaining:",
    invalid_cancel_check.sum()
)

Invalid cancellation dates remaining: 0


In [21]:
# Fix invalid cancellation dates

invalid_cancel = (
    subscriptions["cancellation_date"].notna()
    & (
        subscriptions["cancellation_date"]
        < subscriptions["subscription_start_date"]
    )
)

print("Invalid cancellation dates:", invalid_cancel.sum())

# Set invalid dates to missing
subscriptions.loc[invalid_cancel, "cancellation_date"] = pd.NaT

print(
    "Invalid cancellation dates after cleaning:",
    (
        subscriptions["cancellation_date"].notna()
        & (
            subscriptions["cancellation_date"]
            < subscriptions["subscription_start_date"]
        )
    ).sum()
)

Invalid cancellation dates: 0
Invalid cancellation dates after cleaning: 0


In [22]:
payments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110735 entries, 0 to 110734
Data columns (total 8 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   payment_id             110735 non-null  object 
 1   customer_id            110735 non-null  object 
 2   subscription_id        0 non-null       float64
 3   payment_date           110735 non-null  object 
 4   amount                 110735 non-null  float64
 5   payment_method         110735 non-null  object 
 6   payment_status         110735 non-null  object 
 7   failed_payment_reason  7206 non-null    object 
dtypes: float64(2), object(6)
memory usage: 6.8+ MB


In [23]:
print("Payment status distribution:")
print(payments["payment_status"].value_counts(dropna=False))

Payment status distribution:
payment_status
Success    103529
Failed       7206
Name: count, dtype: int64


In [24]:
print("Failed payment reasons by payment status:")
print(
    payments.groupby("payment_status")["failed_payment_reason"]
    .apply(lambda x: x.notna().sum())
)

Failed payment reasons by payment status:
payment_status
Failed     7206
Success       0
Name: failed_payment_reason, dtype: int64


In [25]:
print("Missing subscription IDs:", payments["subscription_id"].isna().sum())
print("Unique subscription IDs:", payments["subscription_id"].nunique())

Missing subscription IDs: 110735
Unique subscription IDs: 0


In [26]:
print("Unique customers in payments:", payments["customer_id"].nunique())
print("Unique customers in subscriptions:", subscriptions["customer_id"].nunique())

print("\nPayment customer IDs missing from subscriptions:")
print(
    (~payments["customer_id"].isin(subscriptions["customer_id"])).sum()
)

Unique customers in payments: 8000
Unique customers in subscriptions: 8000

Payment customer IDs missing from subscriptions:
0


In [27]:
payments["payment_date"] = pd.to_datetime(
    payments["payment_date"],
    errors="coerce"
)

print("Invalid payment dates:", payments["payment_date"].isna().sum())
print("Payment date range:")
print(payments["payment_date"].min(), "to", payments["payment_date"].max())

Invalid payment dates: 0
Payment date range:
2019-01-01 00:00:00 to 2026-05-31 00:00:00


In [28]:
print("Payment amount statistics:")
print(payments["amount"].describe())

print("\nZero amount payments:",
      (payments["amount"] == 0).sum())

print("Negative amount payments:",
      (payments["amount"] < 0).sum())

Payment amount statistics:
count    110735.000000
mean         13.750196
std           9.510979
min           0.000000
25%           6.990000
50%          15.490000
75%          15.490000
max         252.890000
Name: amount, dtype: float64

Zero amount payments: 250
Negative amount payments: 0


In [29]:
zero_payments = payments[payments["amount"] == 0]

print("Zero-value payments by status:")
print(zero_payments["payment_status"].value_counts())

print("\nZero-value payments by payment method:")
print(zero_payments["payment_method"].value_counts())

Zero-value payments by status:
payment_status
Success    239
Failed      11
Name: count, dtype: int64

Zero-value payments by payment method:
payment_method
Credit Card      73
UPI              49
Debit Card       49
PayPal           24
Mobile Wallet    22
Net Banking      21
Gift Card        12
Name: count, dtype: int64


In [30]:
print("Total payment records:", len(payments))
print("Unique payment IDs:", payments["payment_id"].nunique())
print("Duplicate payment IDs:",
      payments["payment_id"].duplicated().sum())

Total payment records: 110735
Unique payment IDs: 110735
Duplicate payment IDs: 0


In [31]:
print("Failed payments without a failure reason:")
print(
    payments[
        (payments["payment_status"] == "Failed") &
        (payments["failed_payment_reason"].isna())
    ].shape[0]
)

print("\nSuccessful payments with a failure reason:")
print(
    payments[
        (payments["payment_status"] == "Success") &
        (payments["failed_payment_reason"].notna())
    ].shape[0]
)

Failed payments without a failure reason:
0

Successful payments with a failure reason:
0


In [32]:
print(payments.dtypes)

payment_id                       object
customer_id                      object
subscription_id                 float64
payment_date             datetime64[ns]
amount                          float64
payment_method                   object
payment_status                   object
failed_payment_reason            object
dtype: object


In [33]:
support_tickets.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6120 entries, 0 to 6119
Data columns (total 11 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ticket_id                    6120 non-null   object 
 1   customer_id                  6120 non-null   object 
 2   ticket_date                  6120 non-null   object 
 3   issue_category               6120 non-null   object 
 4   issue_subcategory            6120 non-null   object 
 5   priority                     6120 non-null   object 
 6   ticket_status                6120 non-null   object 
 7   resolution_time_hours        4592 non-null   float64
 8   customer_satisfaction_score  4592 non-null   float64
 9   ticket_description           6120 non-null   object 
 10  support_agent_id             6120 non-null   object 
dtypes: float64(2), object(9)
memory usage: 526.1+ KB


In [34]:
print("Ticket status distribution:")
print(support_tickets["ticket_status"].value_counts(dropna=False))

Ticket status distribution:
ticket_status
Resolved     3390
Closed       1202
Pending       888
Escalated     640
Name: count, dtype: int64


In [35]:
print("Missing resolution time by ticket status:")
print(
    support_tickets.groupby("ticket_status")["resolution_time_hours"]
    .apply(lambda x: x.isna().sum())
)

print("\nMissing satisfaction score by ticket status:")
print(
    support_tickets.groupby("ticket_status")["customer_satisfaction_score"]
    .apply(lambda x: x.isna().sum())
)

Missing resolution time by ticket status:
ticket_status
Closed         0
Escalated    640
Pending      888
Resolved       0
Name: resolution_time_hours, dtype: int64

Missing satisfaction score by ticket status:
ticket_status
Closed         0
Escalated    640
Pending      888
Resolved       0
Name: customer_satisfaction_score, dtype: int64


In [36]:
support_tickets["ticket_date"] = pd.to_datetime(
    support_tickets["ticket_date"],
    errors="coerce"
)

print("Invalid ticket dates:",
      support_tickets["ticket_date"].isna().sum())

print("\nTicket date range:")
print(
    support_tickets["ticket_date"].min(),
    "to",
    support_tickets["ticket_date"].max()
)

Invalid ticket dates: 61

Ticket date range:
2019-03-03 00:00:00 to 2026-06-30 00:00:00


In [37]:
raw_support_tickets = pd.read_csv(RAW_DIR / "support_tickets.csv")

invalid_ticket_dates = support_tickets["ticket_date"].isna()

print("Invalid ticket date rows:", invalid_ticket_dates.sum())

print("\nOriginal ticket date values:")
print(
    raw_support_tickets.loc[
        invalid_ticket_dates,
        ["ticket_id", "customer_id", "ticket_date", "ticket_status"]
    ]
)

Invalid ticket date rows: 61

Original ticket date values:
       ticket_id customer_id ticket_date ticket_status
167   TCK0000168  CUST000196  2025-13-45     Escalated
233   TCK0000234  CUST000293  2025-13-45      Resolved
296   TCK0000297  CUST000372  2025-13-45      Resolved
319   TCK0000320  CUST000397  2025-13-45      Resolved
346   TCK0000347  CUST000428  2025-13-45       Pending
...          ...         ...         ...           ...
5153  TCK0005154  CUST006716  2025-13-45      Resolved
5164  TCK0005165  CUST006737  2025-13-45      Resolved
5339  TCK0005340  CUST006968  2025-13-45       Pending
5446  TCK0005447  CUST007133  2025-13-45      Resolved
5858  TCK0005859  CUST007663  2025-13-45       Pending

[61 rows x 4 columns]


In [38]:
print("Invalid ticket dates after cleaning:",
      support_tickets["ticket_date"].isna().sum())

Invalid ticket dates after cleaning: 61


In [39]:
print("Total ticket records:", len(support_tickets))
print("Unique ticket IDs:", support_tickets["ticket_id"].nunique())
print(
    "Duplicate ticket IDs:",
    support_tickets["ticket_id"].duplicated().sum()
)

Total ticket records: 6120
Unique ticket IDs: 6120
Duplicate ticket IDs: 0


In [40]:
print("Resolution time statistics:")
print(support_tickets["resolution_time_hours"].describe())

print("\nNegative resolution times:",
      (support_tickets["resolution_time_hours"] < 0).sum())

print("\nSatisfaction score statistics:")
print(support_tickets["customer_satisfaction_score"].describe())

print("\nSatisfaction scores outside 1-5:",
      (
          (support_tickets["customer_satisfaction_score"] < 1) |
          (support_tickets["customer_satisfaction_score"] > 5)
      ).sum())

Resolution time statistics:
count    4592.000000
mean       17.998911
std        17.689077
min         0.500000
25%         5.200000
50%        12.700000
75%        25.000000
max       151.100000
Name: resolution_time_hours, dtype: float64

Negative resolution times: 0

Satisfaction score statistics:
count    4592.000000
mean        3.258493
std         1.107013
min         1.000000
25%         3.000000
50%         3.000000
75%         4.000000
max         5.000000
Name: customer_satisfaction_score, dtype: float64

Satisfaction scores outside 1-5: 0


In [41]:
unknown_customers = (
    ~support_tickets["customer_id"].isin(customers["customer_id"])
)

print("Tickets with unknown customer IDs:", unknown_customers.sum())

Tickets with unknown customer IDs: 0


In [42]:
viewing_activity.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66422 entries, 0 to 66421
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   viewing_id                66422 non-null  object 
 1   customer_id               66422 non-null  object 
 2   content_id                66422 non-null  object 
 3   viewing_date              66422 non-null  object 
 4   watch_duration_minutes    66422 non-null  int64  
 5   completion_percentage     65891 non-null  float64
 6   device_type               65758 non-null  object 
 7   login_location            66422 non-null  object 
 8   session_duration_minutes  66422 non-null  int64  
dtypes: float64(1), int64(2), object(6)
memory usage: 4.6+ MB


In [43]:
print("Completion percentage statistics:")
print(viewing_activity["completion_percentage"].describe())

Completion percentage statistics:
count    65891.000000
mean        70.474283
std         20.141997
min          1.000000
25%         57.000000
50%         71.000000
75%         86.000000
max        100.000000
Name: completion_percentage, dtype: float64


In [44]:
missing_completion = viewing_activity[
    viewing_activity["completion_percentage"].isna()
]

print("Records with missing completion percentage:",
      len(missing_completion))

print("\nWatch duration for missing completion:")
print(missing_completion["watch_duration_minutes"].describe())

Records with missing completion percentage: 531

Watch duration for missing completion:
count    531.000000
mean      47.824859
std       32.044448
min        3.000000
25%       29.500000
50%       46.000000
75%       64.000000
max      546.000000
Name: watch_duration_minutes, dtype: float64


In [45]:
invalid_completion = (
    (viewing_activity["completion_percentage"] < 0) |
    (viewing_activity["completion_percentage"] > 100)
)

print("Invalid completion percentages:",
      invalid_completion.sum())

Invalid completion percentages: 0


In [46]:
completion_median = viewing_activity["completion_percentage"].median()

viewing_activity["completion_percentage"] = (
    viewing_activity["completion_percentage"]
    .fillna(completion_median)
)

print("Median completion percentage used:", completion_median)
print(
    "Missing completion percentages after cleaning:",
    viewing_activity["completion_percentage"].isna().sum()
)

Median completion percentage used: 71.0
Missing completion percentages after cleaning: 0


In [47]:
print("Device type distribution:")
print(viewing_activity["device_type"].value_counts(dropna=False))

Device type distribution:
device_type
Smart TV            21143
Mobile              16706
Laptop               7606
Tablet               6668
Gaming Console       3350
SmartTV              1021
smart tv             1010
SMART TV              995
TV                    963
MOBILE                802
Smartphone            796
Phone                 760
mobile                758
NaN                   664
TABLET                391
tablet                380
LAPTOP                377
iPad                  372
Desktop               361
laptop                350
PC                    340
Console               210
PlayStation/Xbox      209
gaming console        190
Name: count, dtype: int64


In [48]:
viewing_activity["device_type"] = (
    viewing_activity["device_type"]
    .str.strip()
    .str.lower()
)

print(viewing_activity["device_type"].value_counts(dropna=False))

device_type
smart tv            23148
mobile              18266
laptop               8333
tablet               7439
gaming console       3540
smarttv              1021
tv                    963
smartphone            796
phone                 760
NaN                   664
ipad                  372
desktop               361
pc                    340
console               210
playstation/xbox      209
Name: count, dtype: int64


In [49]:
# Fill missing device types
viewing_activity["device_type"] = viewing_activity["device_type"].fillna("unknown")

# Verify
print("Missing device types after cleaning:")
print(viewing_activity["device_type"].isna().sum())

print("\nDevice type distribution after cleaning:")
print(viewing_activity["device_type"].value_counts())

Missing device types after cleaning:
0

Device type distribution after cleaning:
device_type
smart tv            23148
mobile              18266
laptop               8333
tablet               7439
gaming console       3540
smarttv              1021
tv                    963
smartphone            796
phone                 760
unknown               664
ipad                  372
desktop               361
pc                    340
console               210
playstation/xbox      209
Name: count, dtype: int64


In [50]:
# Convert viewing_date to datetime
viewing_activity["viewing_date"] = pd.to_datetime(
    viewing_activity["viewing_date"],
    errors="coerce"
)

# Check invalid/missing dates
print("Invalid viewing dates:")
print(viewing_activity["viewing_date"].isna().sum())

# Check date range
print("\nViewing date range:")
print(viewing_activity["viewing_date"].min())
print(viewing_activity["viewing_date"].max())

Invalid viewing dates:
0

Viewing date range:
2025-07-01 00:00:00
2026-06-30 00:00:00


In [51]:
print("Watch duration statistics:")
print(viewing_activity["watch_duration_minutes"].describe())

print("\nNegative watch durations:")
print((viewing_activity["watch_duration_minutes"] < 0).sum())

print("\nSession duration statistics:")
print(viewing_activity["session_duration_minutes"].describe())

print("\nNegative session durations:")
print((viewing_activity["session_duration_minutes"] < 0).sum())

Watch duration statistics:
count    66422.000000
mean        49.433862
std         40.388162
min          3.000000
25%         31.000000
50%         48.000000
75%         64.000000
max        894.000000
Name: watch_duration_minutes, dtype: float64

Negative watch durations:
0

Session duration statistics:
count    66422.000000
mean        56.922285
std         40.800459
min          1.000000
25%         38.000000
50%         55.000000
75%         73.000000
max        907.000000
Name: session_duration_minutes, dtype: float64

Negative session durations:
0


In [52]:
# Check whether watch duration exceeds session duration
invalid_duration = (
    viewing_activity["watch_duration_minutes"]
    > viewing_activity["session_duration_minutes"]
)

print("Watch duration greater than session duration:")
print(invalid_duration.sum())

Watch duration greater than session duration:
6079


In [53]:
# Inspect records where watch duration exceeds session duration
invalid_duration_rows = viewing_activity[
    viewing_activity["watch_duration_minutes"]
    > viewing_activity["session_duration_minutes"]
]

print("Number of invalid duration records:", len(invalid_duration_rows))

print("\nSample invalid records:")
print(
    invalid_duration_rows[
        [
            "viewing_id",
            "customer_id",
            "content_id",
            "watch_duration_minutes",
            "completion_percentage",
            "session_duration_minutes"
        ]
    ].head(10)
)

Number of invalid duration records: 6079

Sample invalid records:
       viewing_id customer_id content_id  watch_duration_minutes  \
10   VIEW00000011  CUST000003   CNT00036                      81   
17   VIEW00000018  CUST000004   CNT00184                      52   
43   VIEW00000044  CUST000007   CNT00491                      94   
44   VIEW00000045  CUST000007   CNT00039                      41   
47   VIEW00000048  CUST000008   CNT00035                      46   
79   VIEW00000080  CUST000011   CNT00085                      92   
80   VIEW00000081  CUST000011   CNT00435                      25   
98   VIEW00000099  CUST000014   CNT00327                      28   
120  VIEW00000121  CUST000018   CNT00279                      55   
128  VIEW00000129  CUST000019   CNT00073                       9   

     completion_percentage  session_duration_minutes  
10                    80.0                        80  
17                    38.0                        50  
43                  

In [54]:
print("\nDifference statistics (watch - session):")
print(
    (
        invalid_duration_rows["watch_duration_minutes"]
        - invalid_duration_rows["session_duration_minutes"]
    ).describe()
)


Difference statistics (watch - session):
count    6079.000000
mean        3.188189
std         2.363785
min         1.000000
25%         1.000000
50%         2.000000
75%         4.000000
max        19.000000
dtype: float64


In [55]:
# Fix watch durations greater than session duration
invalid_duration_mask = (
    viewing_activity["watch_duration_minutes"]
    > viewing_activity["session_duration_minutes"]
)

print("Invalid duration records before cleaning:", invalid_duration_mask.sum())

# Cap watch duration at session duration
viewing_activity.loc[
    invalid_duration_mask,
    "watch_duration_minutes"
] = viewing_activity.loc[
    invalid_duration_mask,
    "session_duration_minutes"
]

# Verify after cleaning
remaining_invalid = (
    viewing_activity["watch_duration_minutes"]
    > viewing_activity["session_duration_minutes"]
).sum()

print("Invalid duration records after cleaning:", remaining_invalid)

Invalid duration records before cleaning: 6079
Invalid duration records after cleaning: 0


In [56]:
# Fix invalid watch durations
invalid_duration_mask = (
    viewing_activity["watch_duration_minutes"]
    > viewing_activity["session_duration_minutes"]
)

print("Invalid duration records before cleaning:", invalid_duration_mask.sum())

# Watch duration cannot exceed session duration
viewing_activity.loc[
    invalid_duration_mask,
    "watch_duration_minutes"
] = viewing_activity.loc[
    invalid_duration_mask,
    "session_duration_minutes"
]

# Verify after cleaning
remaining_invalid = (
    viewing_activity["watch_duration_minutes"]
    > viewing_activity["session_duration_minutes"]
).sum()

print("Invalid duration records after cleaning:", remaining_invalid)

Invalid duration records before cleaning: 0
Invalid duration records after cleaning: 0


In [57]:
print("Invalid duration records after cleaning:",
      (viewing_activity["watch_duration_minutes"] >
       viewing_activity["session_duration_minutes"]).sum())

Invalid duration records after cleaning: 0


In [58]:
print("Duplicate viewing IDs:", viewing_activity["viewing_id"].duplicated().sum())
print("Total viewing records:", len(viewing_activity))
print("Unique viewing IDs:", viewing_activity["viewing_id"].nunique())

Duplicate viewing IDs: 0
Total viewing records: 66422
Unique viewing IDs: 66422


In [59]:
print("Unique customers in viewing activity:",
      viewing_activity["customer_id"].nunique())

print("Unique content IDs:",
      viewing_activity["content_id"].nunique())

print("Missing customer IDs:",
      viewing_activity["customer_id"].isna().sum())

print("Missing content IDs:",
      viewing_activity["content_id"].isna().sum())

Unique customers in viewing activity: 7980
Unique content IDs: 500
Missing customer IDs: 0
Missing content IDs: 0


In [60]:
print("Duplicate viewing dates:", viewing_activity["viewing_date"].duplicated().sum())

print("Invalid viewing dates:",
      viewing_activity["viewing_date"].isna().sum())

print("Viewing date range:",
      viewing_activity["viewing_date"].min(),
      "to",
      viewing_activity["viewing_date"].max())

Duplicate viewing dates: 66057
Invalid viewing dates: 0
Viewing date range: 2025-07-01 00:00:00 to 2026-06-30 00:00:00


In [61]:
unknown_viewing_customers = (
    ~viewing_activity["customer_id"].isin(customers["customer_id"])
)

print(
    "Viewing records with unknown customer IDs:",
    unknown_viewing_customers.sum()
)

Viewing records with unknown customer IDs: 0


In [62]:
unknown_content = (
    ~viewing_activity["content_id"].isin(content["content_id"])
)

print(
    "Viewing records with unknown content IDs:",
    unknown_content.sum()
)

print(
    "Unique unknown content IDs:",
    viewing_activity.loc[
        unknown_content, "content_id"
    ].nunique()
)

Viewing records with unknown content IDs: 0
Unique unknown content IDs: 0


In [63]:
print("Missing values after cleaning:")
print(viewing_activity.isna().sum())

print("\nInvalid duration relationships:",
      (
          viewing_activity["watch_duration_minutes"]
          > viewing_activity["session_duration_minutes"]
      ).sum())

print("\nShape:", viewing_activity.shape)

Missing values after cleaning:
viewing_id                  0
customer_id                 0
content_id                  0
viewing_date                0
watch_duration_minutes      0
completion_percentage       0
device_type                 0
login_location              0
session_duration_minutes    0
dtype: int64

Invalid duration relationships: 0

Shape: (66422, 9)


In [64]:
# Save cleaned viewing activity
viewing_activity.to_csv(
    CLEANED_DIR / "viewing_activity.csv",
    index=False
)

print("Saved:", CLEANED_DIR / "viewing_activity.csv")

NameError: name 'CLEANED_DIR' is not defined

In [ ]:
from pathlib import Path

# Create cleaned data folder
CLEANED_DIR = Path("../data/cleaned")
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

# Save cleaned viewing activity
viewing_activity.to_csv(
    CLEANED_DIR / "viewing_activity.csv",
    index=False
)

print("Saved:", CLEANED_DIR / "viewing_activity.csv")

Saved: ..\data\cleaned\viewing_activity.csv


In [ ]:
print("File exists:", (CLEANED_DIR / "viewing_activity.csv").exists())

File exists: True


In [ ]:
content.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   content_id          500 non-null    object
 1   title               500 non-null    object
 2   content_type        500 non-null    object
 3   genre               500 non-null    object
 4   release_year        500 non-null    int64 
 5   duration_minutes    500 non-null    int64 
 6   content_language    500 non-null    object
 7   production_country  500 non-null    object
 8   maturity_rating     500 non-null    object
dtypes: int64(2), object(7)
memory usage: 35.3+ KB


In [ ]:
print("Total content records:", len(content))
print("Unique content IDs:", content["content_id"].nunique())
print("Duplicate content IDs:", content["content_id"].duplicated().sum())

Total content records: 500
Unique content IDs: 500
Duplicate content IDs: 0


In [ ]:
print("Missing values:")
print(content.isna().sum())

print("\nRelease year statistics:")
print(content["release_year"].describe())

print("\nDuration statistics:")
print(content["duration_minutes"].describe())

print("\nInvalid release years:", (content["release_year"] < 1900).sum())
print("Invalid durations:", (content["duration_minutes"] <= 0).sum())

Missing values:
content_id            0
title                 0
content_type          0
genre                 0
release_year          0
duration_minutes      0
content_language      0
production_country    0
maturity_rating       0
dtype: int64

Release year statistics:
count     500.000000
mean     2012.034000
std         8.226311
min      1998.000000
25%      2005.000000
50%      2012.000000
75%      2020.000000
max      2026.000000
Name: release_year, dtype: float64

Duration statistics:
count    500.000000
mean      69.574000
std       35.748853
min       20.000000
25%       37.750000
50%       66.000000
75%      102.000000
max      166.000000
Name: duration_minutes, dtype: float64

Invalid release years: 0
Invalid durations: 0


In [ ]:
# Standardize text columns
text_columns = [
    "title",
    "content_type",
    "genre",
    "content_language",
    "production_country",
    "maturity_rating"
]

for col in text_columns:
    content[col] = content[col].astype(str).str.strip()

print("Text columns standardized.")

Text columns standardized.


In [ ]:
print("Missing values after cleaning:")
print(content.isna().sum())

print("\nShape:", content.shape)
print("\nDuplicate content IDs:", content["content_id"].duplicated().sum())

Missing values after cleaning:
content_id            0
title                 0
content_type          0
genre                 0
release_year          0
duration_minutes      0
content_language      0
production_country    0
maturity_rating       0
dtype: int64

Shape: (500, 9)

Duplicate content IDs: 0


In [ ]:
from pathlib import Path

# Create cleaned data folder
CLEANED_DIR = Path("../data/cleaned")
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

# Save cleaned content data
content.to_csv(
    CLEANED_DIR / "content.csv",
    index=False
)

print("Saved:", CLEANED_DIR / "content.csv")
print("File exists:", (CLEANED_DIR / "content.csv").exists())

Saved: ..\data\cleaned\content.csv
File exists: True


In [65]:
# Load churn labels
churn_labels = pd.read_csv("../data/raw/churn_labels.csv")

print(churn_labels.info())
print("\nFirst 5 rows:")
print(churn_labels.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   customer_id   8000 non-null   object
 1   churned       8000 non-null   bool  
 2   churn_date    1991 non-null   object
 3   churn_reason  1991 non-null   object
dtypes: bool(1), object(3)
memory usage: 195.4+ KB
None

First 5 rows:
  customer_id  churned  churn_date   churn_reason
0  CUST000001    False         NaN            NaN
1  CUST000002    False         NaN            NaN
2  CUST000003     True  2026-04-20  Too expensive
3  CUST000004    False         NaN            NaN
4  CUST000005    False         NaN            NaN


In [66]:
print("\nMissing values:")
print(churn_labels.isna().sum())

print("\nShape:", churn_labels.shape)


Missing values:
customer_id        0
churned            0
churn_date      6009
churn_reason    6009
dtype: int64

Shape: (8000, 4)


In [67]:
print("Churn distribution:")
print(churn_labels["churned"].value_counts())

print("\nChurn date missing by status:")
print(
    churn_labels.groupby("churned")["churn_date"]
    .apply(lambda x: x.isna().sum())
)

print("\nChurn reason missing by status:")
print(
    churn_labels.groupby("churned")["churn_reason"]
    .apply(lambda x: x.isna().sum())
)

Churn distribution:
churned
False    6009
True     1991
Name: count, dtype: int64

Churn date missing by status:
churned
False    6009
True        0
Name: churn_date, dtype: int64

Churn reason missing by status:
churned
False    6009
True        0
Name: churn_reason, dtype: int64


In [68]:
churn_labels["churn_date"] = pd.to_datetime(
    churn_labels["churn_date"],
    errors="coerce"
)

print("Invalid churn dates:",
      churn_labels["churn_date"].isna().sum()
      - (churn_labels["churned"] == False).sum())

print(
    "Churn date range:",
    churn_labels["churn_date"].min(),
    "to",
    churn_labels["churn_date"].max()
)

Invalid churn dates: 0
Churn date range: 2025-10-03 00:00:00 to 2026-06-30 00:00:00


In [69]:
# Convert customer registration dates to datetime
customers["registration_date"] = pd.to_datetime(
    customers["registration_date"],
    errors="coerce"
)

# Compare churn date with registration date
churn_check = churn_labels.merge(
    customers[["customer_id", "registration_date"]],
    on="customer_id",
    how="left"
)

invalid_churn_order = (
    churn_check["churned"]
    & (churn_check["churn_date"] < churn_check["registration_date"])
)

print(
    "Churn dates before registration:",
    invalid_churn_order.sum()
)

Churn dates before registration: 0


In [70]:
print("Total churn label records:", len(churn_labels))
print("Unique customer IDs:", churn_labels["customer_id"].nunique())
print(
    "Duplicate customer IDs:",
    churn_labels["customer_id"].duplicated().sum()
)

Total churn label records: 8000
Unique customer IDs: 8000
Duplicate customer IDs: 0


In [71]:
unknown_churn_customers = (
    ~churn_labels["customer_id"].isin(customers["customer_id"])
)

print(
    "Churn records with unknown customer IDs:",
    unknown_churn_customers.sum()
)

Churn records with unknown customer IDs: 0


In [72]:
# Final validation
print("Shape:", churn_labels.shape)
print("\nMissing values:")
print(churn_labels.isna().sum())
print("\nDuplicate customer IDs:",
      churn_labels["customer_id"].duplicated().sum())

Shape: (8000, 4)

Missing values:
customer_id        0
churned            0
churn_date      6009
churn_reason    6009
dtype: int64

Duplicate customer IDs: 0


In [73]:
# Save cleaned churn labels
CLEANED_DIR = Path("../data/cleaned")
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

churn_labels.to_csv(
    CLEANED_DIR / "churn_labels.csv",
    index=False
)

print("Saved:", CLEANED_DIR / "churn_labels.csv")
print("File exists:", (CLEANED_DIR / "churn_labels.csv").exists())

Saved: ..\data\cleaned\churn_labels.csv
File exists: True


In [74]:
customer_feedback.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5046 entries, 0 to 5045
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   feedback_id      5046 non-null   object
 1   customer_id      5046 non-null   object
 2   feedback_date    5046 non-null   object
 3   rating           5046 non-null   int64 
 4   feedback_text    5046 non-null   object
 5   sentiment_label  5046 non-null   object
dtypes: int64(1), object(5)
memory usage: 236.7+ KB


In [75]:
customer_feedback["rating"].value_counts().sort_index()

rating
1     725
2     711
3    1599
4    1032
5     979
Name: count, dtype: int64

In [76]:
customer_feedback["feedback_date"] = pd.to_datetime(
    customer_feedback["feedback_date"],
    errors="coerce"
)

print("Invalid feedback dates:",
      customer_feedback["feedback_date"].isna().sum())

print("Date range:",
      customer_feedback["feedback_date"].min(),
      "to",
      customer_feedback["feedback_date"].max())

Invalid feedback dates: 0
Date range: 2019-03-06 00:00:00 to 2026-06-30 00:00:00


In [77]:
print("Total feedback records:",
      len(customer_feedback))

print("Unique feedback IDs:",
      customer_feedback["feedback_id"].nunique())

print("Duplicate feedback IDs:",
      customer_feedback["feedback_id"].duplicated().sum())

Total feedback records: 5046
Unique feedback IDs: 5046
Duplicate feedback IDs: 0


In [78]:
unknown_customers = ~customer_feedback["customer_id"].isin(
    customers["customer_id"]
)

print("Unknown customer IDs:",
      unknown_customers.sum())

Unknown customer IDs: 0


In [79]:
print("Empty feedback:",
      customer_feedback["feedback_text"].isna().sum())

print("Blank feedback:",
      (customer_feedback["feedback_text"].str.strip() == "").sum())

Empty feedback: 0
Blank feedback: 0


In [80]:
customer_feedback[[
    "feedback_id",
    "customer_id",
    "rating",
    "feedback_text"
]].head(10)

,feedback_id,customer_id,rating,feedback_text
0,FB0000001,CUST000001,1,"Not enough new content being added, feels like..."
1,FB0000002,CUST000001,5,"Great value for the price, my whole family use..."
2,FB0000003,CUST000002,1,Customer support took too long to respond to m...
3,FB0000004,CUST000003,3,"Average experience overall, some months are be..."
4,FB0000005,CUST000008,3,The service is decent but the content library ...
5,FB0000006,CUST000009,4,"Great value for the price, my whole family use..."
6,FB0000007,CUST000011,1,The app keeps crashing on my device and I have...
7,FB0000008,CUST000013,4,The streaming quality has been excellent and t...
8,FB0000009,CUST000014,3,"It works fine most of the time, occasional buf..."
9,FB0000010,CUST000018,5,"Love the recommendation engine, it keeps sugge..."


In [81]:
customer_feedback["feedback_text"] = (
    customer_feedback["feedback_text"]
    .str.strip()
)

print("Blank feedback after cleaning:",
      (customer_feedback["feedback_text"] == "").sum())

Blank feedback after cleaning: 0


In [82]:
customer_feedback["sentiment_label"].value_counts()

sentiment_label
Positive    1978
Neutral     1606
Negative    1462
Name: count, dtype: int64

In [83]:
print("Shape:", customer_feedback.shape)

print("\nMissing values:")
print(customer_feedback.isna().sum())

print("\nDuplicate feedback IDs:",
      customer_feedback["feedback_id"].duplicated().sum())

Shape: (5046, 6)

Missing values:
feedback_id        0
customer_id        0
feedback_date      0
rating             0
feedback_text      0
sentiment_label    0
dtype: int64

Duplicate feedback IDs: 0


In [84]:
CLEANED_DIR = Path("../data/cleaned")
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

customer_feedback.to_csv(
    CLEANED_DIR / "customer_feedback.csv",
    index=False
)

print("Saved:", CLEANED_DIR / "customer_feedback.csv")
print("File exists:",
      (CLEANED_DIR / "customer_feedback.csv").exists())

Saved: ..\data\cleaned\customer_feedback.csv
File exists: True


In [85]:
subscription_plans.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   plan_id         3 non-null      object 
 1   plan_name       3 non-null      object 
 2   monthly_price   3 non-null      float64
 3   video_quality   3 non-null      object 
 4   max_devices     3 non-null      int64  
 5   advertisements  3 non-null      object 
dtypes: float64(1), int64(1), object(4)
memory usage: 272.0+ bytes


In [86]:
print("Total plans:",
      len(subscription_plans))

print("Unique plan IDs:",
      subscription_plans["plan_id"].nunique())

print("Duplicate plan IDs:",
      subscription_plans["plan_id"].duplicated().sum())

Total plans: 3
Unique plan IDs: 3
Duplicate plan IDs: 0


In [87]:
print(subscription_plans)

  plan_id plan_name  monthly_price    video_quality  max_devices  \
0  PLAN01     Basic           6.99        720p (HD)            1   
1  PLAN02  Standard          15.49  1080p (Full HD)            2   
2  PLAN03   Premium          22.99    4K (Ultra HD)            4   

  advertisements  
0            Yes  
1             No  
2             No  


In [88]:
print("Minimum price:",
      subscription_plans["monthly_price"].min())

print("Maximum price:",
      subscription_plans["monthly_price"].max())

print("Negative prices:",
      (subscription_plans["monthly_price"] < 0).sum())

Minimum price: 6.99
Maximum price: 22.99
Negative prices: 0


In [89]:
print("Minimum devices:",
      subscription_plans["max_devices"].min())

print("Maximum devices:",
      subscription_plans["max_devices"].max())

print("Invalid device values:",
      (subscription_plans["max_devices"] <= 0).sum())

Minimum devices: 1
Maximum devices: 4
Invalid device values: 0


In [90]:
print(subscription_plans["advertisements"].value_counts())

advertisements
No     2
Yes    1
Name: count, dtype: int64


In [91]:
print("Shape:", subscription_plans.shape)

print("\nMissing values:")
print(subscription_plans.isna().sum())

print("\nDuplicate plan IDs:",
      subscription_plans["plan_id"].duplicated().sum())

Shape: (3, 6)

Missing values:
plan_id           0
plan_name         0
monthly_price     0
video_quality     0
max_devices       0
advertisements    0
dtype: int64

Duplicate plan IDs: 0


In [92]:
subscription_plans.to_csv(
    CLEANED_DIR / "subscription_plans.csv",
    index=False
)

print("Saved:", CLEANED_DIR / "subscription_plans.csv")
print("File exists:",
      (CLEANED_DIR / "subscription_plans.csv").exists())

Saved: ..\data\cleaned\subscription_plans.csv
File exists: True


In [93]:
print("Cleaned files:")

for file in CLEANED_DIR.iterdir():
    print(file.name)

Cleaned files:
churn_labels.csv
content.csv
customer_feedback.csv
subscription_plans.csv
viewing_activity.csv


In [94]:
customers.to_csv(
    CLEANED_DIR / "customers.csv",
    index=False
)

print("Saved:", CLEANED_DIR / "customers.csv")
print("File exists:",
      (CLEANED_DIR / "customers.csv").exists())

Saved: ..\data\cleaned\customers.csv
File exists: True


In [95]:
payments.to_csv(
    CLEANED_DIR / "payments.csv",
    index=False
)

print("Saved:", CLEANED_DIR / "payments.csv")
print("File exists:",
      (CLEANED_DIR / "payments.csv").exists())

Saved: ..\data\cleaned\payments.csv
File exists: True


In [96]:
subscriptions.to_csv(
    CLEANED_DIR / "subscriptions.csv",
    index=False
)

print("Saved:", CLEANED_DIR / "subscriptions.csv")
print("File exists:",
      (CLEANED_DIR / "subscriptions.csv").exists())

Saved: ..\data\cleaned\subscriptions.csv
File exists: True


In [97]:
support_tickets.to_csv(
    CLEANED_DIR / "support_tickets.csv",
    index=False
)

print("Saved:", CLEANED_DIR / "support_tickets.csv")
print("File exists:",
      (CLEANED_DIR / "support_tickets.csv").exists())

Saved: ..\data\cleaned\support_tickets.csv
File exists: True


In [98]:
print("Number of cleaned files:",
      len(list(CLEANED_DIR.glob("*.csv"))))

print("\nCleaned datasets:")

for file in CLEANED_DIR.glob("*.csv"):
    print(file.name)

Number of cleaned files: 9

Cleaned datasets:
churn_labels.csv
content.csv
customers.csv
customer_feedback.csv
payments.csv
subscriptions.csv
subscription_plans.csv
support_tickets.csv
viewing_activity.csv
